In [1]:
import sys
!"{sys.executable}" -m pip install -U pip
!"{sys.executable}" -m pip install numpy pandas sqlalchemy "psycopg[binary]" python-dotenv ipykernel

In [8]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine


# НАСТРОЙКИ
CONN_STR = os.environ.get("DATABASE_URL", "").strip()
if not CONN_STR:
    raise RuntimeError(
        "Не найдена переменная окружения DATABASE_URL.\n"
    )


ENGINE_URL = CONN_STR.replace("postgresql://", "postgresql+psycopg://")

SAMPLE_MOD = 5  # ~5% строк, чтобы ускорить bootstrap
MIN_DURATION_MIN = 1
MAX_DURATION_MIN = 180

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)



# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
def bootstrap_diff_mean(a: np.ndarray, b: np.ndarray, n_boot: int = 2000) -> dict:
    """
    Bootstrap разницы средних: mean(a) - mean(b).
    Возвращает эффект + 95% доверительный интервал.
    """
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if len(a) < 50 or len(b) < 50:
        return {
            "diff_mean": float(np.mean(a) - np.mean(b)) if len(a) and len(b) else np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "n_a": int(len(a)),
            "n_b": int(len(b)),
            "note": "Слишком маленькая выборка для bootstrap (нужно хотя бы ~50 на группу).",
        }

    n_a, n_b = len(a), len(b)

    # индексы для бутстрапа: многократно выбираем элементы с возвращением
    idx_a = np.random.randint(0, n_a, size=(n_boot, n_a))
    idx_b = np.random.randint(0, n_b, size=(n_boot, n_b))

    # распределение разницы средних по бутстрап-выборкам
    boot_diff = a[idx_a].mean(axis=1) - b[idx_b].mean(axis=1)

    diff_mean = float(np.mean(a) - np.mean(b))  # разница средних на исходных данных
    ci_low, ci_high = np.percentile(boot_diff, [2.5, 97.5])

    return {
        "diff_mean": diff_mean,
        "ci95_low": float(ci_low),
        "ci95_high": float(ci_high),
        "n_a": int(n_a),
        "n_b": int(n_b),
        "note": "",
    }


def print_section(title: str):
    """Разделитель для вывода."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def safe_print_df(df: pd.DataFrame, n=10):
    """Печать df без обрезки колонок."""
    with pd.option_context("display.max_columns", 200, "display.width", 200):
        print(df.head(n))


# ПОДКЛЮЧЕНИЕ
engine = create_engine(ENGINE_URL)


# SANITY CHECK
print_section("SANITY CHECK: диапазон дат + доли NULL/0 для чаевых")

q_sanity = """
select
  count(*) as n,
  min(tpep_pickup_datetime) as min_dt,
  max(tpep_pickup_datetime) as max_dt,
  avg(case when credit_card_tip_amount is null then 1.0 else 0.0 end) as share_tip_null,
  avg(case when credit_card_tip_amount = 0 then 1.0 else 0.0 end) as share_tip_zero
from public.trips;
"""
sanity = pd.read_sql(q_sanity, engine)
safe_print_df(sanity, 5)

print("\nВажно: у вас чаевые в поле credit_card_tip_amount — это чаевые по карте.")
print("Гипотезы про чаевые корректно сравнивать либо внутри card-платежей,")
print("либо как вероятность tip>0 по типам оплаты (H2 / H5).")


# ГИПОТЕЗА 1

print_section("ГИПОТЕЗА 1: Чаевые выше вечером и в выходные (по карте)")

q_h1_summary = """
with base as (
  select
    extract(dow from tpep_pickup_datetime) as dow,
    extract(hour from tpep_pickup_datetime) as hod,
    credit_card_tip_amount as tip,
    fare_amount as fare,
    credit_card_tip_amount / nullif(fare_amount, 0) as tip_pct
  from public.trips
  where credit_card_tip_amount is not null
    and credit_card_tip_amount >= 0
    and fare_amount > 0
)
select
  case when hod between 18 and 23 then 'evening_18_23' else 'other' end as time_bucket,
  case when dow in (0,6) then 'weekend' else 'weekday' end as day_bucket,
  count(*) as n_trips,
  avg(tip) as avg_tip_amount,
  avg(tip_pct) as avg_tip_pct,
  percentile_cont(0.5) within group (order by tip_pct) as median_tip_pct
from base
group by 1,2
order by 2,1;
"""
h1_summary = pd.read_sql(q_h1_summary, engine)
safe_print_df(h1_summary, 20)

q_h1_sample = f"""
with sample as (
  select
    extract(dow from tpep_pickup_datetime) as dow,
    extract(hour from tpep_pickup_datetime) as hod,
    credit_card_tip_amount / nullif(fare_amount, 0) as tip_pct
  from public.trips
  where credit_card_tip_amount is not null
    and credit_card_tip_amount >= 0
    and fare_amount > 0
    and (extract(epoch from tpep_pickup_datetime)::bigint %% 100) < {SAMPLE_MOD}
)
select
  case when hod between 18 and 23 then 'evening' else 'other' end as time_bucket,
  case when dow in (0,6) then 'weekend' else 'weekday' end as day_bucket,
  tip_pct
from sample
where tip_pct is not null
  and tip_pct between 0 and 1;
"""
h1_sample = pd.read_sql(q_h1_sample, engine)

evening = h1_sample.loc[h1_sample["time_bucket"] == "evening", "tip_pct"].to_numpy(dtype=float)
other_t = h1_sample.loc[h1_sample["time_bucket"] == "other", "tip_pct"].to_numpy(dtype=float)
weekend = h1_sample.loc[h1_sample["day_bucket"] == "weekend", "tip_pct"].to_numpy(dtype=float)
weekday = h1_sample.loc[h1_sample["day_bucket"] == "weekday", "tip_pct"].to_numpy(dtype=float)

print("\n[H1] tip_pct: evening vs other (bootstrap)")
print(bootstrap_diff_mean(evening, other_t, n_boot=2000))

print("\n[H1] tip_pct: weekend vs weekday (bootstrap)")
print(bootstrap_diff_mean(weekend, weekday, n_boot=2000))


# ГИПОТЕЗА 2

print_section("ГИПОТЕЗА 2: Способ оплаты влияет на чаевые")

q_h2 = """
select
  payment_type,
  count(*) as n_trips,
  avg(case when credit_card_tip_amount > 0 then 1.0 else 0.0 end) as p_tip_positive,
  avg(coalesce(credit_card_tip_amount, 0)) as avg_tip_including_zeros,
  avg(coalesce(credit_card_tip_amount, 0) / nullif(fare_amount, 0)) as avg_tip_pct_including_zeros
from public.trips
where fare_amount > 0
group by 1
order by n_trips desc;
"""
h2 = pd.read_sql(q_h2, engine)
safe_print_df(h2, 50)

print("\nКомментарий к H2:")
print("- Метрика чаевых - credit_card_tip_amount.")
print("- Для payment_type, где чаевые не фиксируются в этом поле, среднее будет ~0.")
print("- Это нормальный результат: тип оплаты связан с зарегистрированными чаевыми.")


# ГИПОТЕЗА 3

print_section("ГИПОТЕЗА 3: В часы пик поездки дольше")

q_h3_summary = f"""
with base as (
  select
    extract(hour from tpep_pickup_datetime) as hod,
    extract(epoch from (tpep_dropoff_datetime - tpep_pickup_datetime))/60.0 as duration_min,
    trip_distance_miles
  from public.trips
  where tpep_dropoff_datetime > tpep_pickup_datetime
    and trip_distance_miles is not null
),
clean as (
  select *
  from base
  where duration_min between {MIN_DURATION_MIN} and {MAX_DURATION_MIN}
)
select
  case
    when hod between 7 and 10 or hod between 16 and 19 then 'peak'
    else 'offpeak'
  end as bucket,
  count(*) as n_trips,
  avg(duration_min) as avg_duration_min,
  percentile_cont(0.5) within group (order by duration_min) as median_duration_min,
  avg(trip_distance_miles) as avg_distance_miles
from clean
group by 1;
"""
h3_summary = pd.read_sql(q_h3_summary, engine)
safe_print_df(h3_summary, 10)

q_h3_sample = f"""
with sample as (
  select
    extract(hour from tpep_pickup_datetime) as hod,
    extract(epoch from (tpep_dropoff_datetime - tpep_pickup_datetime))/60.0 as duration_min
  from public.trips
  where tpep_dropoff_datetime > tpep_pickup_datetime
    and (extract(epoch from tpep_pickup_datetime)::bigint %% 100) < {SAMPLE_MOD}
),
clean as (
  select *
  from sample
  where duration_min between {MIN_DURATION_MIN} and {MAX_DURATION_MIN}
)
select
  case
    when hod between 7 and 10 or hod between 16 and 19 then 'peak'
    else 'offpeak'
  end as bucket,
  duration_min
from clean;
"""
h3_sample = pd.read_sql(q_h3_sample, engine)

peak_d = h3_sample.loc[h3_sample["bucket"] == "peak", "duration_min"].to_numpy(dtype=float)
off_d = h3_sample.loc[h3_sample["bucket"] == "offpeak", "duration_min"].to_numpy(dtype=float)

print("\n[H3] duration_min: peak vs offpeak (bootstrap)")
print(bootstrap_diff_mean(peak_d, off_d, n_boot=2000))


print_section("ДОП: Скорость в пик ниже? (avg_speed_mph)")

q_speed = """
with base as (
  select
    extract(hour from tpep_pickup_datetime) as hod,
    extract(epoch from (tpep_dropoff_datetime - tpep_pickup_datetime))/3600.0 as duration_h,
    trip_distance_miles
  from public.trips
  where tpep_dropoff_datetime > tpep_pickup_datetime
    and trip_distance_miles > 0
),
clean as (
  select *
  from base
  where duration_h between (1.0/60.0) and 3.0
)
select
  case
    when hod between 7 and 10 or hod between 16 and 19 then 'peak'
    else 'offpeak'
  end as bucket,
  count(*) as n_trips,
  avg(trip_distance_miles / nullif(duration_h, 0)) as avg_speed_mph
from clean
group by 1;
"""
speed = pd.read_sql(q_speed, engine)
safe_print_df(speed, 10)



# ГИПОТЕЗА 4: ТОП pickup_borough/zone меняется по времени суток

print_section("ГИПОТЕЗА 4: ТОП pickup_borough/zone меняется по времени суток")

q_h4_top_zones = """
with base as (
  select
    dd.hour as hour_of_day,
    dl.borough as pickup_borough,
    dl.zone as pickup_zone
  from public.fact_trip ft
  join public.dim_datetime dd
    on dd.datetime_id = ft.pickup_datetime_id
  join public.dim_location dl
    on dl.location_id = ft.pickup_location_id
),
bucketed as (
  select
    case
      when hour_of_day between 0 and 5 then 'night_00_05'
      when hour_of_day between 6 and 11 then 'morning_06_11'
      when hour_of_day between 12 and 17 then 'day_12_17'
      else 'evening_18_23'
    end as time_bucket,
    pickup_borough,
    pickup_zone
  from base
),
ranked as (
  select
    time_bucket,
    pickup_borough,
    pickup_zone,
    count(*) as trips,
    row_number() over (partition by time_bucket order by count(*) desc) as rn
  from bucketed
  group by 1,2,3
)
select
  time_bucket,
  pickup_borough,
  pickup_zone,
  trips
from ranked
where rn <= 10
order by time_bucket, trips desc;
"""
h4_top = pd.read_sql(q_h4_top_zones, engine)
safe_print_df(h4_top, 50)

q_h4_top1 = """
with base as (
  select
    dd.hour as hour_of_day,
    dl.borough as pickup_borough,
    dl.zone as pickup_zone
  from public.fact_trip ft
  join public.dim_datetime dd
    on dd.datetime_id = ft.pickup_datetime_id
  join public.dim_location dl
    on dl.location_id = ft.pickup_location_id
),
bucketed as (
  select
    case
      when hour_of_day between 0 and 5 then 'night_00_05'
      when hour_of_day between 6 and 11 then 'morning_06_11'
      when hour_of_day between 12 and 17 then 'day_12_17'
      else 'evening_18_23'
    end as time_bucket,
    pickup_borough,
    pickup_zone
  from base
),
ranked as (
  select
    time_bucket,
    pickup_borough,
    pickup_zone,
    count(*) as trips,
    row_number() over (partition by time_bucket order by count(*) desc) as rn
  from bucketed
  group by 1,2,3
)
select time_bucket, pickup_borough, pickup_zone, trips
from ranked
where rn = 1
order by time_bucket;
"""
h4_top1 = pd.read_sql(q_h4_top1, engine)
print("\n[H4] ТОП-1 зона в каждой группе времени:")
safe_print_df(h4_top1, 10)


# ГИПОТЕЗА 5: passenger_count влияет на вероятность чаевых (tip_amount > 0)

print_section("ГИПОТЕЗА 5: Passenger_count влияет на вероятность чаевых (tip>0)")

q_h5_tip_prob = """
select
  passenger_count,
  count(*) as n_trips,
  avg(case when tip_amount > 0 then 1.0 else 0.0 end) as p_tip_positive,
  avg(tip_amount) as avg_tip_amount,
  avg(tip_amount / nullif(fare_amount, 0)) as avg_tip_pct
from public.fact_trip
where fare_amount > 0
group by passenger_count
order by passenger_count;
"""
h5 = pd.read_sql(q_h5_tip_prob, engine)
safe_print_df(h5, 50)

q_h5_sample = f"""
with sample as (
  select
    passenger_count,
    case when tip_amount > 0 then 1.0 else 0.0 end as tip_positive
  from public.fact_trip
  where fare_amount > 0
    and passenger_count is not null
    and (trip_id %% 100) < {SAMPLE_MOD}  -- берем небольшую подвыборку для bootstrap
)
select
  case when passenger_count = 1 then 'pax_1' else 'pax_2plus' end as pax_group,
  tip_positive
from sample;
"""
h5_sample = pd.read_sql(q_h5_sample, engine)

pax1 = h5_sample.loc[h5_sample["pax_group"] == "pax_1", "tip_positive"].to_numpy(dtype=float)
pax2p = h5_sample.loc[h5_sample["pax_group"] == "pax_2plus", "tip_positive"].to_numpy(dtype=float)

print("\n[H5] Разница вероятности чаевых: pax_1 vs pax_2plus (bootstrap)")
print(bootstrap_diff_mean(pax1, pax2p, n_boot=2000))


SANITY CHECK: диапазон дат + доли NULL/0 для чаевых
       n                    min_dt                    max_dt  share_tip_null  share_tip_zero
0  81873 2015-01-01 00:09:48+00:00 2015-01-31 23:57:12+00:00             0.0        0.401805

Важно: у вас чаевые в поле credit_card_tip_amount — это чаевые по карте.
Гипотезы про чаевые корректно сравнивать либо внутри card-платежей,
либо как вероятность tip>0 по типам оплаты (H2 / H5).

ГИПОТЕЗА 1: Чаевые выше вечером и в выходные (по карте)
     time_bucket day_bucket  n_trips  avg_tip_amount  avg_tip_pct  median_tip_pct
0  evening_18_23    weekday    20185        1.668106     0.143782        0.181818
1          other    weekday    38038        1.555653     0.128284        0.142857
2  evening_18_23    weekend     8238        1.458430     0.128915        0.148148
3          other    weekend    15377        1.413730     0.122098        0.133333

[H1] tip_pct: evening vs other (bootstrap)
{'diff_mean': 0.019532143299993726, 'ci95_low': 0.0113